# UBSN Abstract Summary

This notebook compiles the final **network-level** and **parcel-level** ranking results
across all ROI analyses

Assumptions used here:
- **SRM network ranking** uses final `step4` shared-space mean TSC accuracy (`shared_mean`)
- **Raw / SRM encoding network rankings** use group-level mean encoding `r`
- **Parcel raw / SRM rankings** use parcel-wise mean voxel encoding `r`
- **Parcel SRM fitting ranking** inherits the parent network's final SRM shared-space TSC score,
  because SRM fitting itself is estimated at the network level rather than parcel-by-parcel


In [ ]:
from pathlib import Path
import json

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import image


# =========================
# 1. Project setup
# =========================

BASE_DIR = Path(r"CHANGE_THIS_TO_YOUR_PROJECT_DIR")
RUNS_DIR = BASE_DIR / "runs"
ENCODING_RUNS_DIR = BASE_DIR / "encoding_runs"
ATLAS_PATH = Path(r"CHANGE_THIS_TO_YOUR_SCHAEFER400_ATLAS_NIFTI")

OUTPUT_DIR = BASE_DIR / "UBSN_abstract_outputs"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
CACHE_DIR = OUTPUT_DIR / "cache"

for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 150

print(f"Base directory: {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


# =========================
# 2. Helpers
# =========================

def write_xlsx(df: pd.DataFrame, path: Path, sheet_name: str = "ranking"):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name=sheet_name)
    except Exception:
        with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
            df.to_excel(writer, index=False, sheet_name=sheet_name)


def save_barplot(
    df: pd.DataFrame,
    y_col: str,
    x_col: str,
    title: str,
    xlabel: str,
    out_path: Path,
    top_n: int | None = None,
    palette: str = "viridis",
    hue_col: str | None = None,
    figsize=(10, 7),
):
    plot_df = df.copy()
    if top_n is not None:
        plot_df = plot_df.head(top_n).copy()
    category_order = plot_df[y_col].tolist()

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(
        data=plot_df,
        y=y_col,
        x=x_col,
        order=category_order,
        hue=hue_col,
        dodge=False,
        palette=palette,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    if hue_col is not None and ax.legend_ is not None:
        ax.legend_.set_title("")
    elif ax.legend_ is not None:
        ax.legend_.remove()
    plt.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.close(fig)


def parse_roi_name(run_dir: Path) -> str:
    return run_dir.name.replace("_400_parcels_cleaned", "")


def collect_network_summary() -> pd.DataFrame:
    rows = []
    for run_dir in sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()]):
        roi = parse_roi_name(run_dir)
        enc_run = ENCODING_RUNS_DIR / f"{roi}_UBSN_ROI_encoding_cleaned"

        step1 = json.loads((run_dir / "step1_setup_roi_cleaned" / "json" / "step1_cleaned_summary.json").read_text(encoding="utf-8"))
        step3a = json.loads((run_dir / "step3a_gridsearch_k_iter" / "json" / "step3a_gridsearch_summary.json").read_text(encoding="utf-8"))
        step4 = json.loads((run_dir / "step4_time_segment_classification" / "json" / "step4_summary.json").read_text(encoding="utf-8"))
        step7a = json.loads((enc_run / "step7a_select_alpha_huthstyle" / "json" / "step7a_summary_huthstyle.json").read_text(encoding="utf-8"))
        step7b = json.loads((enc_run / "step7b_raw_bold_encoding" / "json" / "step7b_summary.json").read_text(encoding="utf-8"))
        step7c = json.loads((enc_run / "step7c_srm_reconstructed_encoding" / "json" / "step7c_summary.json").read_text(encoding="utf-8"))
        step7d_subj = pd.read_csv(enc_run / "step7d_statistical_summary_raw_vs_srm" / "csv" / "subject_level_inference_raw_vs_srm.csv")
        step7d_vox = pd.read_csv(enc_run / "step7d_statistical_summary_raw_vs_srm" / "csv" / "fig3_voxel_delta_hist_subjectmean_one_tailed_test.csv")
        step8 = json.loads((enc_run / "step8_brain_mapping_nilearn" / "json" / "step8_summary.json").read_text(encoding="utf-8"))

        rows.append({
            "roi": roi,
            "roi_target": f"{roi}_only",
            "run_name": run_dir.name,
            "encoding_run_name": enc_run.name,
            "n_selected_parcels": step1["n_selected_parcels"],
            "selected_parcels": ",".join(str(x) for x in step1["selected_parcels"]),
            "n_subjects": step1["n_subjects_retained"],
            "n_voxels": step1["common_voxel_count"],
            "srm_best_k": step3a["best_by_shared_mean"]["k"],
            "srm_best_n_iter": step3a["best_by_shared_mean"]["n_iter"],
            "srm_anatomical_mean_tsc": step4["anat_mean"],
            "srm_shared_mean_tsc": step4["shared_mean"],
            "srm_tsc_delta_mean": step4["delta_mean"],
            "srm_subject_fraction_shared_better": step4["subject_fraction_shared_better"],
            "raw_group_mean_r": step7b["group_results"]["mean_r"],
            "raw_group_median_r": step7b["group_results"]["median_r"],
            "raw_group_positive_frac": step7b["group_results"]["positive_frac"],
            "raw_subject_mean_r": step7b["individual_results"]["mean_r_across_subjects"],
            "srm_group_mean_r": step7c["group_results_srm"]["mean_r"],
            "srm_group_median_r": step7c["group_results_srm"]["median_r"],
            "srm_group_positive_frac": step7c["group_results_srm"]["positive_frac"],
            "srm_subject_mean_r": step7c["individual_results_srm"]["mean_r_across_subjects"],
            "encoding_best_alpha": step7a["best_alpha"],
            "encoding_subject_delta_mean_r": float(step7d_subj["mean_delta"].iloc[0]),
            "encoding_subject_delta_median_r": float(step7d_subj["median_delta"].iloc[0]),
            "subjects_improved": int(step7d_subj["subjects_improved"].iloc[0]),
            "pct_improved": float(step7d_subj["pct_improved"].iloc[0]),
            "voxel_positive_fraction": float(step7d_vox["positive_voxel_fraction"].iloc[0]),
            "voxel_delta_mean_r": float(step7d_vox["mean_delta_r"].iloc[0]),
            "step8_raw_group_mean_r": step8["map_summaries"]["raw_group_corr_mean"],
            "step8_srm_group_mean_r": step8["map_summaries"]["srm_group_corr_mean"],
            "step8_delta_group_mean_r": step8["map_summaries"]["delta_group_corr_mean"],
        })

    df = pd.DataFrame(rows).sort_values("roi").reset_index(drop=True)
    return df


def build_example_bold_ref(step1_dir: Path):
    import nibabel as nib

    step1_cfg = json.loads((step1_dir / "step1_config.json").read_text(encoding="utf-8"))
    manifest_df = pd.read_csv(step1_dir / "csv" / "subject_file_manifest.csv")
    retained_df = pd.read_csv(step1_dir / "csv" / "subjects_retained.csv")
    example_subject = retained_df["subject"].iloc[0]
    example_bold_path = Path(manifest_df.loc[manifest_df["subject"] == example_subject, "bold_path"].iloc[0])

    bold_nii = nib.load(str(example_bold_path))
    bold_data = bold_nii.get_fdata(dtype=np.float32)

    story_start_tr = int(step1_cfg["story_start_tr"])
    story_end_tr = int(step1_cfg["story_end_tr"])
    bold_crop = bold_data[..., story_start_tr:story_end_tr]
    bold_ref_nii = nib.Nifti1Image(bold_crop[..., 0], bold_nii.affine, bold_nii.header)

    return step1_cfg, bold_ref_nii


def collect_parcel_summary(network_df: pd.DataFrame) -> pd.DataFrame:
    atlas_nii = nib.load(str(ATLAS_PATH))
    network_lookup = network_df.set_index("roi").to_dict("index")
    parcel_rows = []

    for run_dir in sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()]):
        roi = parse_roi_name(run_dir)
        enc_run = ENCODING_RUNS_DIR / f"{roi}_UBSN_ROI_encoding_cleaned"
        step1_dir = run_dir / "step1_setup_roi_cleaned"

        step1_cfg, bold_ref_nii = build_example_bold_ref(step1_dir)
        selected_label_df = pd.read_csv(step1_dir / "csv" / "selected_parcel_labels.csv")
        selected_parcels = step1_cfg["selected_parcels"]

        atlas_resampled = image.resample_to_img(
            atlas_nii,
            bold_ref_nii,
            interpolation="nearest",
            force_resample=True,
            copy_header=True,
        )
        atlas_labels = np.rint(atlas_resampled.get_fdata()).astype(int)

        raw_img = nib.load(str(enc_run / "step8_brain_mapping_nilearn" / "nii" / "raw_encoding_r_group.nii.gz"))
        srm_img = nib.load(str(enc_run / "step8_brain_mapping_nilearn" / "nii" / "srm_encoding_r_group.nii.gz"))
        delta_img = nib.load(str(enc_run / "step8_brain_mapping_nilearn" / "nii" / "delta_encoding_r_group_srm_minus_raw.nii.gz"))

        raw_data = np.asarray(raw_img.get_fdata(), dtype=float)
        srm_data = np.asarray(srm_img.get_fdata(), dtype=float)
        delta_data = np.asarray(delta_img.get_fdata(), dtype=float)

        for _, label_row in selected_label_df.iterrows():
            parcel_id = int(label_row["parcel_id"])
            parcel_name = str(label_row["parcel_name"])
            mask = atlas_labels == parcel_id
            voxel_count = int(mask.sum())
            if voxel_count == 0:
                continue

            raw_vals = raw_data[mask]
            srm_vals = srm_data[mask]
            delta_vals = delta_data[mask]

            raw_vals = raw_vals[np.isfinite(raw_vals)]
            srm_vals = srm_vals[np.isfinite(srm_vals)]
            delta_vals = delta_vals[np.isfinite(delta_vals)]

            parcel_rows.append({
                "roi": roi,
                "parcel_id": parcel_id,
                "parcel_name": parcel_name,
                "voxel_count": voxel_count,
                "srm_shared_mean_tsc": network_lookup[roi]["srm_shared_mean_tsc"],
                "srm_tsc_delta_mean": network_lookup[roi]["srm_tsc_delta_mean"],
                "raw_group_mean_r": float(np.mean(raw_vals)),
                "srm_group_mean_r": float(np.mean(srm_vals)),
                "delta_group_mean_r": float(np.mean(delta_vals)),
            })

    parcel_df = pd.DataFrame(parcel_rows).sort_values(["roi", "parcel_id"]).reset_index(drop=True)
    return parcel_df


def build_subject_level_tables() -> tuple[pd.DataFrame, pd.DataFrame]:
    enc_rows = []
    tsc_rows = []

    for run_dir in sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()]):
        roi = parse_roi_name(run_dir)
        enc_run = ENCODING_RUNS_DIR / f"{roi}_UBSN_ROI_encoding_cleaned"

        raw_df = pd.read_csv(enc_run / "step7b_raw_bold_encoding" / "csv" / "individual_encoding_summary_raw.csv")
        srm_df = pd.read_csv(enc_run / "step7c_srm_reconstructed_encoding" / "csv" / "individual_encoding_summary_srm.csv")
        tsc_df = pd.read_csv(run_dir / "step4_time_segment_classification" / "csv" / "subjectwise_time_segment_classification.csv")

        for _, row in raw_df.iterrows():
            enc_rows.append({
                "roi": roi,
                "subject_index": int(row["subject_index"]),
                "condition": "raw",
                "mean_r": float(row["mean_r"]),
                "median_r": float(row["median_r"]),
                "positive_frac": float(row["positive_frac"]),
            })
        for _, row in srm_df.iterrows():
            enc_rows.append({
                "roi": roi,
                "subject_index": int(row["subject_index"]),
                "condition": "srm",
                "mean_r": float(row["mean_r"]),
                "median_r": float(row["median_r"]),
                "positive_frac": float(row["positive_frac"]),
            })

        for _, row in tsc_df.iterrows():
            tsc_rows.append({
                "roi": roi,
                "subject_index": int(row["subject_index"]),
                "condition": "anatomical",
                "accuracy": float(row["anat_accuracy"]),
            })
            tsc_rows.append({
                "roi": roi,
                "subject_index": int(row["subject_index"]),
                "condition": "shared",
                "accuracy": float(row["shared_accuracy"]),
            })

    return pd.DataFrame(enc_rows), pd.DataFrame(tsc_rows)


# =========================
# 3. Collect summary data
# =========================

network_df = collect_network_summary()
network_df["encoding_group_delta_mean_r"] = (
    network_df["srm_group_mean_r"] - network_df["raw_group_mean_r"]
)
parcel_df = collect_parcel_summary(network_df)
encoding_subject_long_df, tsc_subject_long_df = build_subject_level_tables()

display(network_df)
display(parcel_df.head())


# =========================
# 4. Export master tables
# =========================

network_master_xlsx = TABLE_DIR / "network_summary_master.xlsx"
parcel_master_xlsx = TABLE_DIR / "parcel_summary_master.xlsx"
enc_subject_long_xlsx = TABLE_DIR / "subject_level_encoding_long.xlsx"
tsc_subject_long_xlsx = TABLE_DIR / "subject_level_tsc_long.xlsx"

write_xlsx(network_df, network_master_xlsx, sheet_name="network_summary")
write_xlsx(parcel_df, parcel_master_xlsx, sheet_name="parcel_summary")
write_xlsx(encoding_subject_long_df, enc_subject_long_xlsx, sheet_name="encoding_long")
write_xlsx(tsc_subject_long_df, tsc_subject_long_xlsx, sheet_name="tsc_long")

network_df.to_csv(TABLE_DIR / "network_summary_master.csv", index=False, encoding="utf-8-sig")
parcel_df.to_csv(TABLE_DIR / "parcel_summary_master.csv", index=False, encoding="utf-8-sig")
encoding_subject_long_df.to_csv(TABLE_DIR / "subject_level_encoding_long.csv", index=False, encoding="utf-8-sig")
tsc_subject_long_df.to_csv(TABLE_DIR / "subject_level_tsc_long.csv", index=False, encoding="utf-8-sig")

print(f"Saved master tables to: {TABLE_DIR}")


# =========================
# 5. Network rankings
# =========================

network_metrics = [
    {
        "metric": "srm_shared_mean_tsc",
        "title": "Network Ranking by SRM Shared-Space TSC Accuracy",
        "xlabel": "Shared-space time-segment classification accuracy",
        "xlsx_name": "network_ranking_srm_shared_tsc.xlsx",
        "png_name": "network_ranking_srm_shared_tsc.png",
    },
    {
        "metric": "raw_group_mean_r",
        "title": "Network Ranking by Raw LLM Encoding Performance",
        "xlabel": "Group-level mean encoding r",
        "xlsx_name": "network_ranking_raw_llm_encoding.xlsx",
        "png_name": "network_ranking_raw_llm_encoding.png",
    },
    {
        "metric": "srm_group_mean_r",
        "title": "Network Ranking by SRM + LLM Encoding Performance",
        "xlabel": "Group-level mean encoding r",
        "xlsx_name": "network_ranking_srm_llm_encoding.xlsx",
        "png_name": "network_ranking_srm_llm_encoding.png",
    },
    {
        "metric": "encoding_group_delta_mean_r",
        "title": "Network Ranking by SRM + LLM Encoding Gain",
        "xlabel": "Group-level mean delta r (SRM + LLM - Raw + LLM)",
        "xlsx_name": "network_ranking_srm_minus_raw_llm_encoding_delta.xlsx",
        "png_name": "network_ranking_srm_minus_raw_llm_encoding_delta.png",
    },
]

for spec in network_metrics:
    metric = spec["metric"]
    rank_df = network_df.sort_values(metric, ascending=False).reset_index(drop=True).copy()
    rank_df.insert(0, "rank", np.arange(1, len(rank_df) + 1))
    write_xlsx(rank_df, TABLE_DIR / spec["xlsx_name"], sheet_name="ranking")
    save_barplot(
        rank_df,
        y_col="roi",
        x_col=metric,
        title=spec["title"],
        xlabel=spec["xlabel"],
        out_path=FIG_DIR / spec["png_name"],
        palette="viridis",
        figsize=(10, 8),
    )
    display(rank_df[["rank", "roi", metric, "n_selected_parcels", "n_voxels"]])


# =========================
# 6. Parcel rankings (Top 20 shown in figures)
# =========================

parcel_metrics = [
    {
        "metric": "srm_shared_mean_tsc",
        "title": "Top 20 Parcels by Parent-Network SRM Shared-Space TSC Accuracy",
        "xlabel": "Shared-space time-segment classification accuracy",
        "xlsx_name": "parcel_ranking_srm_shared_tsc.xlsx",
        "png_name": "parcel_top20_srm_shared_tsc.png",
    },
    {
        "metric": "raw_group_mean_r",
        "title": "Top 20 Parcels by Raw LLM Encoding Performance",
        "xlabel": "Parcel-level mean encoding r",
        "xlsx_name": "parcel_ranking_raw_llm_encoding.xlsx",
        "png_name": "parcel_top20_raw_llm_encoding.png",
    },
    {
        "metric": "srm_group_mean_r",
        "title": "Top 20 Parcels by SRM + LLM Encoding Performance",
        "xlabel": "Parcel-level mean encoding r",
        "xlsx_name": "parcel_ranking_srm_llm_encoding.xlsx",
        "png_name": "parcel_top20_srm_llm_encoding.png",
    },
]

for spec in parcel_metrics:
    metric = spec["metric"]
    rank_df = parcel_df.sort_values([metric, "voxel_count"], ascending=[False, False]).reset_index(drop=True).copy()
    rank_df.insert(0, "rank", np.arange(1, len(rank_df) + 1))
    rank_df["parcel_label"] = rank_df["parcel_id"].astype(str) + " | " + rank_df["parcel_name"]
    write_xlsx(rank_df, TABLE_DIR / spec["xlsx_name"], sheet_name="ranking")
    save_barplot(
        rank_df,
        y_col="parcel_label",
        x_col=metric,
        title=spec["title"],
        xlabel=spec["xlabel"],
        out_path=FIG_DIR / spec["png_name"],
        top_n=20,
        palette="magma",
        hue_col="roi",
        figsize=(14, 9),
    )
    display(rank_df[["rank", "roi", "parcel_id", "parcel_name", metric, "voxel_count"]].head(20))


# =========================
# 7. Quick cross-metric views
# =========================

# Each point in these scatterplots is one ROI/network.
# We compare network-level summary metrics that were already computed above:
# 1. step4 SRM shared-space TSC accuracy
# 2. step7b raw group-level mean encoding r
# 3. step7c SRM-reconstructed group-level mean encoding r
# 4. step7d subject-level mean delta r
cross_metric_df = network_df[[
    "roi",
    "srm_shared_mean_tsc",
    "raw_group_mean_r",
    "srm_group_mean_r",
    "encoding_group_delta_mean_r",
    "encoding_subject_delta_mean_r",
]].copy()

write_xlsx(cross_metric_df, TABLE_DIR / "network_cross_metric_scatterplot_source.xlsx", sheet_name="cross_metric")
cross_metric_df.to_csv(TABLE_DIR / "network_cross_metric_scatterplot_source.csv", index=False, encoding="utf-8-sig")

roi_order = cross_metric_df["roi"].tolist()
palette = dict(zip(roi_order, sns.color_palette("tab20", n_colors=len(roi_order))))

fig, axes = plt.subplots(1, 2, figsize=(16, 6.8))
sns.scatterplot(
    data=cross_metric_df,
    x="srm_shared_mean_tsc",
    y="srm_group_mean_r",
    hue="roi",
    palette=palette,
    s=120,
    ax=axes[0],
)
axes[0].set_title("SRM TSC Accuracy vs SRM + LLM Encoding Performance", pad=18)
axes[0].set_xlabel("Shared-space time-segment classification accuracy")
axes[0].set_ylabel("Group-level mean encoding r")

sns.scatterplot(
    data=cross_metric_df,
    x="raw_group_mean_r",
    y="srm_group_mean_r",
    hue="roi",
    palette=palette,
    s=120,
    ax=axes[1],
)
axes[1].set_title("Raw vs SRM + LLM Encoding Performance", pad=18)
axes[1].set_xlabel("Raw group-level mean encoding r")
axes[1].set_ylabel("SRM group-level mean encoding r")
if axes[0].legend_ is not None:
    axes[0].legend_.remove()
handles, labels = axes[1].get_legend_handles_labels()
if axes[1].legend_ is not None:
    axes[1].legend_.remove()

fig.legend(
    handles,
    labels,
    title="Network",
    bbox_to_anchor=(0.98, 0.5),
    loc="center right",
    frameon=False,
)

plt.subplots_adjust(wspace=0.32, right=0.82, top=0.88)
plt.savefig(FIG_DIR / "network_cross_metric_scatterplots.png", bbox_inches="tight")
plt.close(fig)

print("All ranking tables and figures have been generated.")


## Network-Level SRM+LLM Gain Similarity Summary

This optional summary figure combines three pieces of information in one circular network plot:

- node size: group-level SRM+LLM encoding performance
- outer radial bar: mean subject-level paired gain, SRM+LLM minus Raw+LLM
- chord/line: FDR-significant participant-wise gain-profile similarity between networks

The chords do not indicate functional connectivity or information flow. They indicate that two networks show similar across-participant SRM-related encoding gain patterns.


In [ ]:
# =========================
# Optional circular network summary:
# SRM+LLM performance, paired gain, and participant-wise gain similarity
# =========================

from itertools import combinations
from math import atan2, cos, sin, pi

try:
    from scipy import stats
except Exception as exc:
    raise ImportError("This cell requires scipy for Pearson correlations and t-tests.") from exc

# This cell assumes the main UBSN_abstract cell above has already produced:
# - network_df
# - encoding_subject_long_df
# - FIG_DIR
# - TABLE_DIR

summary_df = network_df.copy()
if "encoding_group_delta_mean_r" not in summary_df.columns:
    summary_df["encoding_group_delta_mean_r"] = summary_df["srm_group_mean_r"] - summary_df["raw_group_mean_r"]

# Build one 25-dimensional paired gain vector per network:
# gain(subject, network) = SRM+LLM mean r - Raw+LLM mean r
subject_gain_df = (
    encoding_subject_long_df
    .pivot_table(
        index=["roi", "subject_index"],
        columns="condition",
        values="mean_r",
        aggfunc="mean",
    )
    .reset_index()
)
subject_gain_df["paired_gain_r"] = subject_gain_df["srm"] - subject_gain_df["raw"]

gain_wide = subject_gain_df.pivot(index="subject_index", columns="roi", values="paired_gain_r")

# Network-level paired gain and two-sided one-sample t-test for star annotations.
# This tests whether participant-wise SRM+LLM - Raw+LLM gains differ from zero.
network_gain_stats = []
for roi in gain_wide.columns:
    vals = gain_wide[roi].dropna().to_numpy(dtype=float)
    t_stat, p_val = stats.ttest_1samp(vals, popmean=0.0, nan_policy="omit")
    network_gain_stats.append({
        "roi": roi,
        "mean_subject_paired_gain_r": float(np.nanmean(vals)),
        "sd_subject_paired_gain_r": float(np.nanstd(vals, ddof=1)),
        "t_stat": float(t_stat),
        "p_value": float(p_val),
    })
def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty_like(adjusted)
    out[order] = adjusted
    return out

network_gain_stats_df = pd.DataFrame(network_gain_stats)
network_gain_stats_df["p_fdr"] = fdr_bh(network_gain_stats_df["p_value"].values)

# Pairwise participant-wise gain-profile correlations between networks.
edge_rows = []
for roi_a, roi_b in combinations(gain_wide.columns, 2):
    pair_df = gain_wide[[roi_a, roi_b]].dropna()
    if len(pair_df) < 4:
        continue
    r_val, p_val = stats.pearsonr(pair_df[roi_a], pair_df[roi_b])
    edge_rows.append({
        "network_a": roi_a,
        "network_b": roi_b,
        "r": float(r_val),
        "p_value": float(p_val),
        "n_subjects": int(len(pair_df)),
    })
edge_df = pd.DataFrame(edge_rows)

# Benjamini-Hochberg FDR correction, implemented locally to keep the cell self-contained.
if len(edge_df) > 0:
    edge_df["p_fdr"] = fdr_bh(edge_df["p_value"].values)
    edge_df["significant_fdr"] = edge_df["p_fdr"] < 0.05
else:
    edge_df["p_fdr"] = []
    edge_df["significant_fdr"] = []

# Keep positive FDR-significant similarities. If there are too many, keep the strongest 30.
plot_edge_df = (
    edge_df
    .query("significant_fdr == True and r > 0")
    .sort_values("r", ascending=False)
    .head(30)
    .copy()
)

# Merge plotting metrics.
plot_df = (
    summary_df[["roi", "srm_group_mean_r", "raw_group_mean_r", "encoding_group_delta_mean_r", "n_selected_parcels", "n_voxels"]]
    .merge(network_gain_stats_df, on="roi", how="left")
)

# Order networks by broad family and then by SRM+LLM encoding performance.
def network_family(roi):
    if roi.startswith("Vis"):
        return "Visual"
    if roi.startswith("SomMot"):
        return "SomMot"
    if roi.startswith("DorsAttn"):
        return "DorsAttn"
    if roi.startswith("SalVentAttn"):
        return "SalVentAttn"
    if roi.startswith("Cont"):
        return "Control"
    if roi.startswith("Default"):
        return "Default"
    if roi.startswith("Limbic"):
        return "Limbic"
    if roi.startswith("TempPar"):
        return "TempPar"
    return "Other"

family_order = ["Visual", "SomMot", "DorsAttn", "SalVentAttn", "Control", "Default", "Limbic", "TempPar", "Other"]
family_colors = {
    "Visual": "#E66101",
    "SomMot": "#1F9ACB",
    "DorsAttn": "#5E3C99",
    "SalVentAttn": "#B2ABD2",
    "Control": "#2CA25F",
    "Default": "#1B7837",
    "Limbic": "#D01C8B",
    "TempPar": "#E7298A",
    "Other": "#777777",
}

plot_df["family"] = plot_df["roi"].map(network_family)
plot_df["family_rank"] = plot_df["family"].map({f: i for i, f in enumerate(family_order)})
plot_df = plot_df.sort_values(["family_rank", "srm_group_mean_r"], ascending=[True, False]).reset_index(drop=True)
rois = plot_df["roi"].tolist()

# Coordinates on a circle.
n = len(rois)
angles = np.linspace(pi / 2, pi / 2 - 2 * pi, n, endpoint=False)
angle_lookup = dict(zip(rois, angles))
xy = {roi: np.array([cos(angle_lookup[roi]), sin(angle_lookup[roi])]) for roi in rois}

# Scaling helpers.
def rescale_values(values, out_min, out_max):
    values = np.asarray(values, dtype=float)
    if np.nanmax(values) == np.nanmin(values):
        return np.repeat((out_min + out_max) / 2, len(values))
    return out_min + (values - np.nanmin(values)) / (np.nanmax(values) - np.nanmin(values)) * (out_max - out_min)

plot_df["node_size"] = rescale_values(plot_df["srm_group_mean_r"], 260, 1300)
plot_df["bar_len"] = rescale_values(plot_df["mean_subject_paired_gain_r"], 0.045, 0.22)
plot_df["color"] = plot_df["family"].map(family_colors)
plot_df["sig_label"] = pd.cut(
    plot_df["p_fdr"],
    bins=[-np.inf, 0.001, 0.01, 0.05, np.inf],
    labels=["***", "**", "*", ""],
).astype(str)

# Export source tables for the figure.
write_xlsx(plot_df, TABLE_DIR / "network_gain_similarity_nodes.xlsx", sheet_name="nodes")
write_xlsx(edge_df.sort_values(["significant_fdr", "r"], ascending=[False, False]), TABLE_DIR / "network_gain_similarity_edges.xlsx", sheet_name="edges")

# Draw figure.
from matplotlib.lines import Line2D
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image

fig, ax = plt.subplots(figsize=(17, 15.5), subplot_kw={"aspect": "equal"})
ax.set_facecolor("white")
fig.patch.set_facecolor("white")

# Chords: participant-wise gain-profile similarity.
if len(plot_edge_df) > 0:
    r_min, r_max = plot_edge_df["r"].min(), plot_edge_df["r"].max()
    for _, edge in plot_edge_df.iterrows():
        a, b = edge["network_a"], edge["network_b"]
        if a not in xy or b not in xy:
            continue
        p0 = xy[a] * 0.72
        p2 = xy[b] * 0.72
        control = np.array([0.0, 0.0])
        t = np.linspace(0, 1, 80)
        curve = ((1 - t) ** 2)[:, None] * p0 + (2 * (1 - t) * t)[:, None] * control + (t ** 2)[:, None] * p2
        width = 0.7 + 5.2 * ((edge["r"] - r_min) / (r_max - r_min + 1e-12))
        ax.plot(
            curve[:, 0],
            curve[:, 1],
            color="#D95F02",
            alpha=0.18 + 0.45 * ((edge["r"] - r_min) / (r_max - r_min + 1e-12)),
            linewidth=width,
            solid_capstyle="round",
            zorder=1,
        )

# Subtle circular guides.
for radius, color, lw in [(0.72, "#DDDDDD", 1.0), (0.91, "#BBBBBB", 1.0), (1.14, "#EEEEEE", 1.0), (1.62, "#F3F3F3", 0.9)]:
    theta = np.linspace(0, 2 * pi, 500)
    ax.plot(radius * np.cos(theta), radius * np.sin(theta), color=color, linewidth=lw, zorder=0)

# Outer bars: subject-level paired gain.
for _, row in plot_df.iterrows():
    roi = row["roi"]
    angle = angle_lookup[roi]
    r0 = 0.94
    r1 = 0.94 + row["bar_len"]
    x0, y0 = r0 * cos(angle), r0 * sin(angle)
    x1, y1 = r1 * cos(angle), r1 * sin(angle)
    ax.plot([x0, x1], [y0, y1], color=row["color"], linewidth=8, solid_capstyle="butt", alpha=0.92, zorder=3)
    if row["sig_label"]:
        xs, ys = (r1 + 0.035) * cos(angle), (r1 + 0.035) * sin(angle)
        ax.text(xs, ys, row["sig_label"], ha="center", va="center", fontsize=10, fontweight="bold", color="#222222", zorder=6)

# Nodes: SRM+LLM group-level performance.
for _, row in plot_df.iterrows():
    roi = row["roi"]
    x, y = xy[roi] * 0.86
    ax.scatter(x, y, s=row["node_size"], color=row["color"], edgecolor="white", linewidth=1.8, zorder=4)
    label_radius = 1.24
    lx, ly = label_radius * cos(angle_lookup[roi]), label_radius * sin(angle_lookup[roi])
    rotation = np.degrees(angle_lookup[roi])
    if rotation < -90:
        rotation += 180
        ha = "right"
    elif rotation > 90:
        rotation -= 180
        ha = "right"
    else:
        ha = "left"
    ax.text(lx, ly, roi, rotation=rotation, ha=ha, va="center", fontsize=9, fontweight="bold", color="#222222", zorder=7)

# Outer brain maps: crop only the 4th glass-brain panel for each network.
# The source glass-brain figure has four panels plus a colorbar; this crop keeps
# the rightmost brain panel and removes the title, other panels, and colorbar.
def crop_fourth_glass_panel(image_path):
    pil_img = Image.open(image_path).convert("RGB")
    width, height = pil_img.size

    # Keep the manually tuned 4th-panel crop parameters unchanged.
    left = int(width * 0.725)
    right = int(width * 0.915)
    top = int(height * 0.105)
    bottom = int(height * 0.885)
    panel = pil_img.crop((left, top, right, bottom))

    # Add the original delta colorbar from the right side of the glass-brain figure.
    cbar_left = int(width * 0.925)
    cbar_right = int(width * 0.999)
    cbar_top = int(height * 0.035)
    cbar_bottom = int(height * 0.930)
    colorbar = pil_img.crop((cbar_left, cbar_top, cbar_right, cbar_bottom))
    colorbar = colorbar.resize((max(10, int(panel.width * 0.25)), panel.height))

    pad = max(10, int(panel.width * 0.055))
    combined = Image.new("RGB", (panel.width + pad + colorbar.width, panel.height), "white")
    combined.paste(panel, (0, 0))
    combined.paste(colorbar, (panel.width + pad, 0))
    return np.asarray(combined)

brain_radius = 1.76
brain_zoom = 0.165
for _, row in plot_df.iterrows():
    roi = row["roi"]
    brain_path = (
        ENCODING_RUNS_DIR
        / f"{roi}_UBSN_ROI_encoding_cleaned"
        / "step8_brain_mapping_nilearn"
        / "figures"
        / "glass_delta_encoding_r_group_srm_minus_raw.png"
    )
    if not brain_path.exists():
        print(f"Missing brain map for {roi}: {brain_path}")
        continue
    img = crop_fourth_glass_panel(brain_path)
    image_box = OffsetImage(img, zoom=brain_zoom)
    x_img, y_img = brain_radius * cos(angle_lookup[roi]), brain_radius * sin(angle_lookup[roi])
    ab = AnnotationBbox(
        image_box,
        (x_img, y_img),
        frameon=False,
        box_alignment=(0.5, 0.5),
        pad=0,
        zorder=2,
    )
    ax.add_artist(ab)

# Legend proxies. Keep these outside the data area to avoid covering labels.
family_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor=color, markeredgecolor="white", markersize=9, label=family)
    for family, color in family_colors.items()
    if family in set(plot_df["family"])
]
size_handles = [
    Line2D([0], [0], marker="o", color="none", markerfacecolor="#888888", markeredgecolor="white", markersize=s, label=lab)
    for s, lab in [(6, "lower SRM+LLM r"), (11, "higher SRM+LLM r")]
]
bar_handle = [Line2D([0], [0], color="#555555", linewidth=6, label="Outer bar = mean subject-level paired gain")]
line_handle = [Line2D([0], [0], color="#D95F02", linewidth=4, alpha=0.55, label="Line = FDR-significant gain-profile similarity")]
brain_handle = [Line2D([0], [0], marker="s", color="none", markerfacecolor="#DDDDDD", markeredgecolor="#888888", markersize=9, label="Outer inset = Δ r (Srm-Raw)")]

leg1 = ax.legend(
    handles=family_handles,
    title="Network",
    loc="upper left",
    bbox_to_anchor=(0.94, 0.76),
    frameon=False,
    fontsize=8.5,
    title_fontsize=10,
)
ax.add_artist(leg1)
leg2 = ax.legend(
    handles=size_handles + bar_handle + line_handle + brain_handle,
    title="Encoding summary",
    loc="upper left",
    bbox_to_anchor=(0.94, 0.56),
    frameon=False,
    fontsize=8.5,
    title_fontsize=10,
)

ax.set_title(
    "Network-Level Summary of SRM-Related LLM Encoding Improvement",
    fontsize=17,
    fontweight="bold",
    pad=24,
)
ax.text(
    0,
    -2.24,
    "Nodes: SRM+LLM group-level mean encoding r | Bars: mean within-subject gain | Lines: FDR-significant participant-wise gain similarity",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#444444",
)
ax.text(
    0,
    -2.36,
    "Outer brain insets: group-level SRM+LLM minus Raw+LLM encoding improvement",
    ha="center",
    va="center",
    fontsize=9.5,
    color="#444444",
)
ax.text(
    0,
    -2.48,
    "* FDR-corrected two-sided one-sample tests of participant-wise paired gain against zero.",
    ha="center",
    va="center",
    fontsize=9.2,
    color="#444444",
)
ax.set_xlim(-2.25, 2.45)
ax.set_ylim(-2.58, 2.20)
ax.axis("off")

out_png = FIG_DIR / "network_srm_llm_gain_similarity_circular_summary.png"
out_pdf = FIG_DIR / "network_srm_llm_gain_similarity_circular_summary.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved circular summary figure: {out_png}")
print(f"Saved circular summary figure: {out_pdf}")
print(f"Saved node source table: {TABLE_DIR / 'network_gain_similarity_nodes.xlsx'}")
print(f"Saved edge source table: {TABLE_DIR / 'network_gain_similarity_edges.xlsx'}")
